# v12 · Prime zero index: Horner base-95 → next prime → π(p)
`VAPMIP/monad.py` `_horner_hash`, `_next_prime`, `_word_zero_idx` (git 2026-05-19; `_word_zero_idx` byte-identical since commit `204c75d`, 2026-05-27).

In [1]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

corpus: 4000 chars, sha256 3105964b9180566a


In [2]:
mo = load("VAPMIP/monad.py", "mo")
for w in ["a", "listen", "silent", "Listen", "hyperwebster"]:
    h = mo._horner_hash(w); p = mo._next_prime(h)
    print("%-13s horner=%-26d next_prime=%-6d zero_idx=%d" % (w, h, p, mo._word_zero_idx(w)))
print("index range:", min(mo._prime_pi_table[2:]), "to", max(mo._prime_pi_table), "| prime cap:", mo._PRIME_CAP)

a             horner=65                         next_prime=67     zero_idx=19
listen        horner=594091334983               next_prime=2879   zero_idx=417
silent        horner=648249864469               next_prime=761    zero_idx=135
Listen        horner=346481434983               next_prime=30161  zero_idx=3262
hyperwebster  horner=414915708702006481311862   next_prime=50671  zero_idx=5195
index range: 1 to 6543 | prime cap: 65536


The Horner integer is order-sensitive (unlike v11's sum), so anagrams get different integers. Two things then make the map many-to-one: `_next_prime` rounds up to the next prime (so neighbouring integers share an index, even for single letters), and it first reduces the integer modulo the prime cap + 1 (a word of three or more characters already exceeds the cap). The result is one of at most 6,543 zero indices.

## Exhaustive collision count: every lowercase word of length ≤ 3

In [3]:
from itertools import product
import string
words = ["".join(p) for n in (1, 2, 3) for p in product(string.ascii_lowercase, repeat=n)]
idx = Counter(mo._word_zero_idx(w) for w in words)
print(len(words), "words ->", len(idx), "distinct zero indices; largest bucket:", max(idx.values()))
assert len(idx) < len(words)          # pigeonhole: 18,278 words, at most 6,542 slots
lens = Counter(len(w) for w in words)
by_len = {n: len({mo._word_zero_idx(w) for w in words if len(w) == n}) for n in (1, 2, 3)}
print("distinct indices by word length:", by_len, "| words by length:", dict(lens))

18278 words -> 2409 distinct zero indices; largest bucket: 34
distinct indices by word length: {1: 7, 2: 101, 3: 2304} | words by length: {1: 26, 2: 676, 3: 17576}


This is the lossy semantic hash the paper keeps separate from the exact bijection: it cannot be inverted (many words per index), and it is used as a *lookup key into the zeros*, never as a storage address.